In [17]:
CREATE MODEL evaluator(
	input
) ENGINE =  OpenAI(api_key = 'sk-proj-xxxx',  model_name = 'gpt-4-turbo')
PROMPT (system "You are an evaluator model. Your task to check if the input provided has atleast these things in order
    You have to ensure that genearated code is SQL only and not any other language
    1 A way to load data: externally by pdf or html for websites 
	2 CREATE Prompt statement
	3 Create Model statement
	4 Quering the model
    Use the tools provided to evaluate the SQL statements
    Make sure that there is SQL code for CREATE PROMPT and CREATE MODEL
    ",
	human "{{input}}"
)
TOOLS ( model_evaluator)
SETTINGS retries = 3;  


In [32]:
CREATE ENDPOINT evaluator(input String) AS 
(SELECT  evaluator($input))
COMMENT 'Agent to call evaluator';

In [19]:
CREATE MODEL model_evaluator(
    input
) ENGINE = OpenAI(api_key = 'sk-proj-xxxx',  model_name = 'gpt-4-turbo')
PROMPT (system "You are an evaluator. Your main task is to check and correct if the SQL code written for quering/calling the model has the following format:
        SELECT model_name('<input>');
    model_name is a placeholder and <input> is also placeholder where the actual input will be there.
    Evalate the statement and correct it if it is wrong",
    human "{{input}}")
SETTINGS retries = 3;  

In [20]:
CREATE ENDPOINT model_evaluator(input String) AS 
(SELECT  model_evaluator($input))
COMMENT 'Agent to evaluate quering the model';

In [18]:
CREATE MODEL prompt_evaluator(
    input
) ENGINE = OpenAI(api_key = 'sk-proj-xxxx',  model_name = 'gpt-4-turbo')
PROMPT (system "You are an evaluator. Your main task is to check and correct if the SQL code written for CREATING PROMPT has the following format:
        CREATE PROMPT (system '<prompt>' ,
        human '{\{model_input}\}')
    <prompt> is a placeholder
    Evalate the statement and correct it if it is wrong",
    human "{{input}}")
SETTINGS retries = 3;  

In [19]:
drop model prompt_evaluator

Successfully Dropped Model: prompt_evaluator

In [21]:
CREATE ENDPOINT html_function() AS
SELECT * from pretty_print((SELECT content from load_html('https://langdb.ai/docs/sql_functions/load_html/')))
COMMENT 'Use this tool to get information regarding function to an html file'


In [22]:
CREATE ENDPOINT pdf_function() AS
SELECT * from pretty_print((SELECT content from load_html('https://langdb.ai/docs/sql_functions/load_pdf_text/')))
COMMENT 'Use this tool to get information regarding function to load pdf file'


In [23]:

CREATE ENDPOINT embed_function() AS
SELECT * from pretty_print((SELECT content from load_html('https://langdb.ai/docs/sql_functions/embed/')))
COMMENT 'Use this tool to get information regarding function to generate embeddings'


In [24]:

CREATE ENDPOINT query_model() AS
SELECT * from pretty_print((SELECT content from load_html('https://langdb.ai/docs/learn/models/')))
COMMENT 'Use this tool to get information regarding function to use the generated model'


In [9]:
CREATE ENDPOINT create_endpoint() AS
SELECT * from pretty_print((SELECT content from load_html('https://langdb.ai/docs/learn/endpoints/')))
COMMENT 'Use this tool to get information regarding how to create endpoints'

In [25]:
CREATE PROMPT function_prompt(
	system 'Use the provided tools to get the functions needed for the task and generate sql for the same.
    Here is the information about different tools available to you:
    1. html_function: Using this tool, you can fetch information on SQL for extracting/loading a webpage. This can be used to get data from an html. URL
    2. pdf_function: Using this tool, you can get information on SQL for loading text from a pdf, Use this for pdf analysis, asking questions over it.
    3. embeded_function: Using this tool, you can get information on SQL for generating vector embeddings
', 
	human '{{task}}' 
);


In [26]:
CREATE MODEL IF NOT EXISTS get_available_functions(
	task
) ENGINE = OpenAI(api_key = 'sk-proj-xxxx',  model_name = 'gpt-4-turbo')
PROMPT function_prompt
TOOLS (
	html_function,
	pdf_function,
	embed_function
	)
SETTINGS retries = 4;

Successfully Dropped Model: get_available_functions

In [27]:
CREATE ENDPOINT IF NOT EXISTS get_available_functions(input String "Describe what you want function to do") AS
(SELECT  get_available_functions($input))
COMMENT 'Use this tool to get list of supported functions in LangDB to load and embed data';

In [28]:
CREATE PROMPT notebook_prompt (
system "
Your task is to help generate a sequence of text and LangDB SQL code blocks for creating a Jupyter Notebook sample. The sequence follows a structured pattern to facilitate tasks such as QA over a PDF document, recommending travel destinations, or other use cases specified by the user.
Given an input. Create a series of document and sql code. 
    
    
I will list down the flow for few usecases
A. Answering over PDF/webpage
1. Load the data either by using pdf or url using required functions 
2. Use Endpoints to create a tool to attach to model
3. Generate embeddings if necessary
4. Create appropriate prompt
5. Generate Model

B. Sentiment Analysis Over Input:
1. Load data if needed using the functions from get_available_functions. Store the data if needed
2. Create prompt : Create prompt for your usecase.
3. Create model : Create Model for your usecase
4. Call the model over the natural language input.

Make use of langdb_generate_sql tool to generate prompt and model statements. 
For langdb_generate_sql tool, you need to pass prompt_name, model_name and prompt.
    
Use the provided tools, To generate sql code.
Make sure to use the evaluator tool to evaulate sql code generated. 

Here is the information about different tools available to you:
1. langdb_generate_sql : Use this tool to get LangDB SQL Code for model and prompt creation. You should pass the following
parameters to this tool which are generated based on the usecase given as input:
 a) prompt_name : Name of the prompt for the usecase model
 b) prompt: Extensive, specific piece of text which prompts the model what to do.
 c) model_name: Name of the model eg. finance_assistant, pdf_assistant etc.
 Generate SQL for CREATE PRONMPT first then CREATE MODEL
 
2. get_available_functions: To generate SQL Code for tasks other than prompt or model creation, use this tool. Pass the 
task to this model : be specific . This tool will return you the SQL Code.
This tool tells about all the sql functions available as well as how to call models. 

3. evaluator: To evaulate generated sql format. Use this to cross check all the generated sql code. 


Use the following format:
    Question: the sample usecase given by the user for which you have to generate the sequence of text and code blocks.
    Text Block : The first step in the sequence of text and code blocks.
    Code Block: The code block corresponding to the text block.
    ... (this Text/Code block can repeat N times)

    Begin!

",

human "{{input}}"
);

In [29]:
CREATE MODEL generate_notebook (
	input
) ENGINE = OpenAI(api_key = 'sk-proj-xxxx',  model_name = 'gpt-4-turbo')
PROMPT notebook_prompt
TOOLS (
	langdb_generate_sql,
	get_available_functions,
    evaluator
)
SETTINGS retries = 3;

In [35]:
select generate_notebook('i want to do sentiment analysis')

Begin!

**Text Block:** The first step in performing sentiment analysis is to determine if data loading is necessary, and if so, which specific functions from LangDB will be used.

**Code Block:**
```sql
SELECT * FROM get_available_functions WHERE task_description LIKE '%load%';
```

**Text Block:** Next, we create a specific prompt to guide the model for sentiment analysis.

**Codeduplicate post:**
```sql
CALL langdb_generate_sql({
  "prompt_name": "sentiment_analysis_prompt",
  "prompt": "Determine the sentiment of the following text input, categorizing it as positive, negative, or neutral."
});
```

**Text Block:** After setting up our prompt, we proceed to create a model specifically designed for sentiment analysis.

**Code Block:**
```sql
CALL langdb_generate_sql({
  "model_name": "sentiment_analysis_model",
  "prompt_name": "sentiment_analysis_prompt"
});
```

**Text Block:** Finally, we can utilize our model to process the text for sentiment analysis.

**Code Block:**
```sql
SEL

In [7]:
select generate_notebook('i want to do sentiment analysis')

To conduct sentiment analysis, we first need to create a model that can classify text into either neutral, negative, or positive sentiments. Here's how you can approach this:

1. **Create the Sentiment Analysis Model**: We'll use an OpenAI model and embed it within a SQL engine to classify text according to sentiment. This operation is integrated directly within a SQL query environment, allowing you to pass natural language text directly to the model for sentiment classification.

2. **Query the Model Over Input**: Once the model is created, you can classify any text by sending it as input to the query model function.

Here is a sample SQL code to first create and then use the sentiment analysis model:

```sql
-- Creating the sentiment analysis model
CREATE MODEL sentiment_analysis(input) ENGINE = OpenAI(api_key='your-api-key', model_name='gpt-3.5-turbo', temperature=0.3) 
PROMPT ("Classify the text into neutral, negative, or positive Text: {{input}} Sentiment:");

-- Example query usi

In [40]:
select generate_notebook('i want to do finacial analysis over a pdf')

### Creating a Jupyter Notebook for Financial Analysis over a PDF

Here's how you can set up a Jupyter Notebook to perform financial analysis on text data extracted from a PDF document.

#### 1. Load the PDF Data
To begin, you'll need to load the content of your PDF into a SQL-readable format. We'll use the `load_pdf_text` function for this. Here's the SQL command to load your PDF data:

```sql
-- Load text content from a PDF file
SELECT *
FROM load_pdf_text('YOUR_PDF_FILE_URL', {
  'chunk_size': 1000,  -- Adjust the chunk size as needed
  'chunk_overlap': 100,  -- Adjust the overlap as needed
  'trim': true  -- Set to true to trim white spaces
});
```
Replace `'YOUR_PDF_FILE_URL'` with the actual URL or file path of your PDF.

#### 2. Create SQL Prompt and Model for Financial Analysis
Here's the SQL code to create a prompt and a model for analyzing the financial data extracted:

```sql
-- Create the prompt for financial analysis
CREATE PROMPT 'financial_analysis_over_pdf' 
(
    syste

In [49]:
select generate_notebook('i want ask questions over a webpage html')

### Step 1: Loading the Data from a Webpage

To facilitate querying and extracting information from a webpage, the first step is to load the HTML content of the page into a structured format that allows easy access and manipulation.

#### Code Block: Load the Data from a Webpage
Let's collect the available functions to load the HTML content of a webpage.### Step 1: Load the Webpage HTML Content

To begin with, we'll use the `load_html()` SQL function to load the HTML from the webpage. Assuming the URL for the webpage is "https://en.wikipedia.org/wiki/New_Delhi", the following SQL code will load its content:

#### Code Block:
```sql
SELECT *
FROM load_html('https://en.wikipedia.org/wiki/New_Delhi');
```

Next, we'll use this loaded content to create a tool that allows us to attach the model for querying.

### Step 2: Creating a Tool to Attach to the Model

We will use the loaded HTML content to create an endpoint which will act as a tool for querying the webpage.

#### Code Block:
```sq

In [50]:
select generate_notebook('i want to do finacial analysis over a pdf')

### Step 1: Loading the PDF

To perform financial analysis, we first need to load the data from the PDF. We will use `load_pdf_text` function which is designed to extract text from a PDF document. Here is the SQL command to load a PDF from a specific URL:

```sql
SELECT * FROM load_pdf_text('https://example.com/financial_report.pdf');
```

### Step 2: Create a Prompt for Financial Analysis

Now we need to define a specific prompt that will guide the analysis over the data extracted from the PDF. We need to create a prompt that specifies what financial aspects to analyze, such as balance sheets, income statements, or cash flow analysis.

### Step 3: Generate SQL for Prompt Creation

Let's generate the SQL for creating the prompt for our financial analysis.

```json
{
  "prompt_name": "financial_analysis_prompt",
  "prompt": "Analyze the given financial data focusing on key performance indicators, profitability, liquidity, and solvency ratios. Summarize insights about financial health an

In [53]:
select generate_notebook('i want to ask questions over a webpage')

Begin!

Text Block: To start answering questions from a webpage, we first need to load the webpage into our database.

Code Block:### Step 1: Load the Webpage Content

We will begin by loading the content from the webpage we would like to analyze. Here, you can specify the URL of the webpage from which you wish to extract information.

#### Code Block:
```sql
SELECT *
FROM load_html('YOUR_WEBPAGE_URL');
```
Replace `'YOUR_WEBIDEPAGE_URL'` with the actual URL of the webpage you are interested in. This SQL code will load the webpage and present its content in a structured format.

### Step 2: Create an Endpoint to Query the Webpage

To enable the ability to ask questions over the loaded webpage, we need to create an endpoint using the data. This will allow the model to interact with the loaded content effectively.

For this purpose, let's define an appropriate model and prompt using the `langdb_generate_sql` tool.

#### Generating Model and Prompt via SQL Tool:
Let's use the `langdb_dest

In [34]:
select generate_notebook('i want to ask questions over a a pdf')

Great! You want to create a workflow to ask questions over a PDF document. Let's outline the necessary steps and generate the corresponding SQL code blocks.

### Step 1: Load the PDF data

First, we need to load the PDF data into our environment, extract the text, and potentially generate embeddings for easier querying.

#### Text Block: Load the PDF document and extract text
We'll use the appropriate available function from LangDB to load the PDF and extract text for further processing.
## Text Block: Load the PDF data
To begin working with the PDF, we first need to load and extract its text content. This can be achieved using the provided SQL function to load the PDF from a specified URL or file path, allowing us to specify options for how the text should be split into manageable chunks if necessary.

## Code Block:
```sql
SELECT *
FROM load_pdf_text('https://path_to_your_pdf.pdf', {
    "split": {
        "chunk_size": 1000,
        "chunk_overlap": 100,
        "trim": true
    }
}